In [0]:
%run ./config

In [0]:
%run ./silver_utils

In [0]:
%run ../connection

In [0]:
# =============================================================================
# NOTEBOOK: 02e_silver_global_stats.py
# LAYER:    Silver
# PURPOSE:  Flatten the global crypto market stats Bronze payload into a clean
#           one-row-per-day Silver table.
#
# SOURCE:   bronze/global_raw      (Delta table, path-based)
# OUTPUT:   silver/global_stats    (Delta table, path-based, MERGE upsert)
#
# GLOBAL PAYLOAD STRUCTURE (from Bronze):
#   payload : {
#     "data": {
#       "total_market_cap"                     : { "usd": 2450000000000.0, ... },
#       "total_volume"                         : { "usd": 98000000000.0, ... },
#       "market_cap_percentage"                : { "btc": 52.3, "eth": 17.1, ... },
#       "active_cryptocurrencies"              : 10547,
#       "markets"                              : 862,
#       "market_cap_change_percentage_24h_usd" : -1.23,
#       "updated_at"                           : 1700000000   ← Unix seconds (NOT ms!)
#     }
#   }
#
# KEY DIFFERENCE FROM OTHER ENDPOINTS:
#   - updated_at is Unix SECONDS (not milliseconds like OHLC/market_chart)
#   - total_market_cap and total_volume are nested dicts keyed by currency
#     We extract only the "usd" key from each
#   - market_cap_percentage is a dict — we extract "btc" and "eth" keys
#   - This endpoint returns ONE ROW of data per API call (global aggregate)
#     so after transformation we expect exactly 1 row per daily run
#
# WHAT THIS NOTEBOOK DOES:
#   1. Read latest Bronze batch
#   2. Extract payload.data struct fields (nested dict access)
#   3. Cast updated_at Unix seconds → TimestampType and DateType
#   4. Cast all numeric fields to correct Silver types
#   5. Add pipeline_run_id and silver_processed_timestamp
#   6. Quality filters (no null stats_date, validate BTC dominance range)
#   7. Dedup on stats_date
#   8. MERGE into silver/global_stats
#   9. OPTIMIZE + Z-ORDER
#
# SILVER TABLE: silver_global_stats
#   Grain: one row per day (global market aggregate)
#   Primary key: stats_date
#
# AUTHOR:  [Your Name]
# VERSION: 1.0.0
# =============================================================================


# =============================================================================
# CELL 1 — SETUP
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, TimestampType, DateType
)

adls_name = "adlsnewhp"
init_silver_config(adls_name)

logger = get_logger("silver_global_stats")

logger.info("=" * 70)
logger.info("Silver: global_stats — START")
logger.info(f"  Run ID    : {SilverConfig.RUN_ID}")
logger.info(f"  Bronze in : {BronzePaths.GLOBAL}")
logger.info(f"  Silver out: {SilverPaths.GLOBAL_STATS}")
logger.info("=" * 70)


In [0]:
# =============================================================================
# CELL 2 — READ BRONZE (latest batch)
# =============================================================================

bronze_df, max_bronze_ts = read_bronze_incremental(
    spark, BronzePaths.GLOBAL , WatermarkPaths.GLOBAL,
    WatermarkPaths.WATERMARK_TABLE, logger
)

assert_required_columns(
    bronze_df,
    required_cols=["payload", "pipeline_run_id"],
    table_name="global_raw",
    logger=logger
)

logger.info(f"  Bronze schema: {bronze_df.schema.simpleString()}")

In [0]:
# =============================================================================
# CELL 3 — EXTRACT AND CAST FIELDS
#
# ACCESSING NESTED STRUCTS:
#   payload.data.total_market_cap.usd   → total market cap in USD
#   payload.data.market_cap_percentage.btc → BTC dominance as a percentage
#
# TIMESTAMP CONVERSION — DIFFERENT FROM OHLC:
#   updated_at is Unix SECONDS (the /global endpoint uses seconds, not ms).
#   from_unixtime() takes seconds directly — no division needed here.
#   (Compare with OHLC and market_chart where we divide by 1000 first.)
# =============================================================================

logger.info("CELL 3: Extracting global stats fields from payload.data")

typed_df = (
    bronze_df
    .select(
        # ── Timestamp & Date ─────────────────────────────────────────────────
        # updated_at is Unix SECONDS — from_unixtime() takes seconds directly
        F.from_unixtime(
            F.col("payload.data.updated_at").cast("long")
        ).cast(TimestampType()).alias("stats_timestamp"),

        F.to_date(
            F.from_unixtime(F.col("payload.data.updated_at").cast("long"))
        ).alias("stats_date"),                          # Primary key

        # ── Global Market Cap ─────────────────────────────────────────────────
        # total_market_cap is a struct with one key per currency (usd, eur, btc, ...)
        # We extract only the USD value
        F.col("payload.data.total_market_cap.usd")
         .cast(DoubleType())
         .alias("total_market_cap_usd"),

        # ── Global 24h Volume ─────────────────────────────────────────────────
        F.col("payload.data.total_volume.usd")
         .cast(DoubleType())
         .alias("total_volume_24h_usd"),

        # ── BTC & ETH Dominance ───────────────────────────────────────────────
        # market_cap_percentage is a struct of {btc: 52.3, eth: 17.1, bnb: 3.2, ...}
        # Values are percentages (52.3 = 52.3%, not 0.523)
        F.col("payload.data.market_cap_percentage.btc")
         .cast(DoubleType())
         .alias("btc_dominance_pct"),

        F.col("payload.data.market_cap_percentage.eth")
         .cast(DoubleType())
         .alias("eth_dominance_pct"),

        # ── Market Breadth ────────────────────────────────────────────────────
        F.col("payload.data.active_cryptocurrencies")
         .cast(IntegerType())
         .alias("active_cryptos"),

        # ── Market Sentiment Indicator ────────────────────────────────────────
        # Negative = market losing value (bearish), positive = gaining (bullish)
        F.col("payload.data.market_cap_change_percentage_24h_usd")
         .cast(DoubleType())
         .alias("market_cap_change_pct_24h"),

        # ── Lineage ───────────────────────────────────────────────────────────
        F.col("pipeline_run_id").cast(StringType()),
    )
    .withColumn("silver_processed_timestamp", get_silver_timestamp(SilverConfig.RUN_TS))
)

raw_count = typed_df.count()
logger.info(f"  Rows after field extraction (expect 1 per run): {raw_count:,}")



In [0]:
# =============================================================================
# CELL 4 — DATA QUALITY FILTERS
#
# GLOBAL-SPECIFIC RULES:
#   - stats_date must not be null (no date = can't place this on timeline)
#   - total_market_cap_usd must be positive (sanity: the global market has value)
#   - btc_dominance_pct must be in [0, 100] (it's a percentage)
#   - active_cryptos must be >= MIN_ACTIVE_CRYPTOS
# =============================================================================

logger.info("CELL 4: Applying data quality filters")

clean_df = (
    typed_df
    .filter(F.col("stats_date").isNotNull())
    .filter(F.col("stats_timestamp").isNotNull())
    .filter(
        F.col("total_market_cap_usd").isNotNull() &
        (F.col("total_market_cap_usd") > 0.0)
    )
    .filter(
        F.col("btc_dominance_pct").isNull() |
        (
            (F.col("btc_dominance_pct") >= DataQuality.MIN_BTC_DOMINANCE_PCT) &
            (F.col("btc_dominance_pct") <= DataQuality.MAX_BTC_DOMINANCE_PCT)
        )
    )
    .filter(
        F.col("active_cryptos").isNull() |
        (F.col("active_cryptos") >= DataQuality.MIN_ACTIVE_CRYPTOS)
    )
)

clean_count = clean_df.count()
logger.info(f"  Rows after quality filters: {clean_count:,}")

validate_drop_rate(
    rows_before  = raw_count,
    rows_after   = clean_count,
    max_fraction = DataQuality.MAX_DROP_FRACTION,
    table_name   = "global_stats",
    logger       = logger,
)

In [0]:
# =============================================================================
# CELL 5 — DEDUPLICATE
# Edge case: if the pipeline ran twice in one day (retry), we keep only one row.
# =============================================================================

logger.info("CELL 5: Deduplicating on stats_date")

dedup_df    = clean_df.dropDuplicates(MergeKeys.GLOBAL_STATS)
dedup_count = dedup_df.count()
logger.info(f"  Rows after dedup: {dedup_count:,} (dropped {clean_count - dedup_count:,})")


In [0]:


# =============================================================================
# CELL 6 — FINAL COLUMN REORDER
# =============================================================================

final_df = dedup_df.select(*SilverColumns.GLOBAL_STATS)
log_schema(final_df, "global_stats_final", logger)

In [0]:
# =============================================================================
# CELL 7 — DELTA MERGE
# =============================================================================

logger.info("CELL 7: MERGE into silver/global_stats")

merge_stats = delta_merge(
    spark      = spark,
    new_df     = final_df,
    table_path = SilverPaths.GLOBAL_STATS,
    merge_keys = MergeKeys.GLOBAL_STATS,
    logger     = logger,
)

# Update watermark ONLY after successful MERGE.
# If MERGE failed, an exception would have already propagated — we never reach here.
update_watermark(
    spark          = spark,
    source_table   = WatermarkPaths.GLOBAL,
    new_ts         = max_bronze_ts,
    watermark_path = WatermarkPaths.WATERMARK_TABLE,
    run_ts         = SilverConfig.RUN_TS,
    logger         = logger,
)
 


In [0]:

# =============================================================================
# CELL 8 — OPTIMIZE + Z-ORDER
# global_stats has only one row per day, so Z-ORDER on stats_date allows Gold
# to skip all but the relevant date's files in date-range queries.
# =============================================================================

logger.info("CELL 8: OPTIMIZE silver/global_stats")

spark.sql(f"OPTIMIZE delta.`{SilverPaths.GLOBAL_STATS}` ZORDER BY (stats_date)")
logger.info("  ✓ OPTIMIZE complete")


In [0]:




# =============================================================================
# CELL 9 — RUN LOG + COMPLETION
# =============================================================================

summary = {
    "notebook"                 : "02e_silver_global_stats",
    "pipeline_run_id"          : SilverConfig.RUN_ID,
    "run_timestamp_utc"        : SilverConfig.RUN_TS.isoformat(),
    "bronze_source"            : BronzePaths.GLOBAL,
    "silver_target"            : SilverPaths.GLOBAL_STATS,
    "rows_from_bronze"         : raw_count,
    "rows_after_quality_filter": clean_count,
    "rows_after_dedup"         : dedup_count,
    "merge_rows_before"        : merge_stats["rows_before"],
    "merge_rows_after"         : merge_stats["rows_after"],
    "merge_rows_inserted"      : merge_stats["rows_inserted"],
    "status"                   : "SUCCESS",
}

write_run_log(summary, LogPaths.GLOBAL_STATS, logger)

logger.info("=" * 70)
logger.info("Silver: global_stats — COMPLETE")
for k, v in summary.items():
    logger.info(f"  {k:<35}: {v}")
logger.info("=" * 70)

dbutils.notebook.exit(json.dumps(summary))
